# Week 4 Day 2: Core Supervised Learning: Preprocessing, Models & Evaluation
### Goal: 
To make repeatable pipelines

Preprocessing the data

Training two supervised models: logistic regression and decision trees

Evaluate on Hold-Out Test

### Dataset (same as before)

We are using the Adult Census Income Dataset.

Some stuff we did yesterday:

* 0 = <=50K
* 1 = >50K

## Imports and Loading


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

#For task 1: preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
adult = fetch_openml(
    name="adult",
    version=2,              #apparently the second version doesnt have any ? so thats why i wasnt able to "clean" that
    as_frame=True
)

df = adult.frame.copy()
df.head()


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [3]:
df.rename(columns={"class": "income"}, inplace=True)
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='str')

In [6]:
df["income"] = df["income"].map({"<=50K" : 0, ">50K" : 1})   #mapping it just like day1

In [8]:
df.duplicated().sum()

np.int64(52)

In [7]:
df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
48838,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
48839,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
48840,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


### Fixing issues from day 1
1. **Missing categories** in workclass, occupation, native_country (~2-6% each)
2. **Categorical encoding**: we might need one-hot encoding for workclass, education,
   marital_status, occupation, relationship, race, sex, native_country before feeding a actual model
3. **Skewed numerics**: capital_gain and capital_loss are extremely skewed
4. **Redundant features**: education (categorical) and education_num (numerical) have the same info so we might wanna keep just one
5. **fnlwgt** can be dropped since it does nothing for us
6. **Age is Nonlinear:** earning power vs. age is not linear, will have to look into that too
7. **? Mystery** Tried to find it but didnt get it, not sure if its an issue tho


Ill do a few of these these in the next few cells during cleaning and preprocessing
Wont touch the 3rd one yet (skewed numerics) since thats not part of today's task. Although we could use `log1p()` to fix it.


## Task 1: Preprocessing Plan & Implementation
We shall sta